# Walmart Sales Analysis (Cleaned Notebook)

Streamlit 대시보드에 반영된 분석 코드만 남기고 재구성했습니다.


## 0) 환경 준비 / 데이터 로드


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


In [ ]:
# 데이터 로드 (대시보드와 동일한 경로 사용)
df = pd.read_csv("./data/walmart_data.csv")
df["Purchase"] = pd.to_numeric(df["Purchase"], errors="coerce")
df.head()


## 1) 데이터 살펴보기

### 1-1) Purchase 분포


In [ ]:
plt.figure(figsize=(10, 5))
df["Purchase"].dropna().hist(bins=40)
plt.title("Purchase Distribution")
plt.xlabel("Purchase")
plt.ylabel("Count")
plt.show()


## 2) 고객 중심 분석

### 2-1) 인사이트 요약

1인당 평균 93개 제품을 사지만 중간값은 54개로, 사업등의 이유로 대량구매를 하는 고객들 때문에 평균이 상승함 <br>
26-35세가 가장 구매력이 높게 나타남. 그 뒤로 36-45, 18-25 순 (구매빈도 및 구매금액이 비슷한 분포를 가짐)

18~45세 연령대가 안정적인 고객군으로 판단됩니다.<br>
10대의 경우 보호자 동반 방문으로 직접 방문하는 고객층의 패턴을 확인할 필요가 있습니다.<br>
50대 이상의 경우 구매력에 비해 방문횟수가 적어, 마케팅 강화의 여지를 확인할 필요가 있습니다.<br>


### 2-2) 고객 통계 테이블 (user_stat_df)


In [ ]:
# 사용자 기본정보(중복 제거) + 유저별 구매 통계 결합
user_df = df.drop_duplicates(subset=["User_ID"], keep="first").copy()
drop_cols = [c for c in ["Product_ID", "Product_Category", "Purchase"] if c in user_df.columns]
user_df = user_df.drop(columns=drop_cols)

user_purchase_df = (
    df.groupby("User_ID")["Purchase"]
    .aggregate(["count", "sum"])
    .rename(columns={"count": "Product_Kinds", "sum": "Purchase_Amount"})
    .reset_index()
)

user_stat_df = user_df.merge(user_purchase_df, on="User_ID", how="left").set_index("User_ID").sort_index()
user_stat_df.head()


### 2-3) 1인당 구매품목 / 1인당 구매금액 분포 (boxplot)


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

axs[0].boxplot(user_stat_df["Product_Kinds"].dropna(), vert=True)
axs[0].set_title("Product Kinds per User")

axs[1].boxplot(user_stat_df["Purchase_Amount"].dropna(), vert=True)
axs[1].set_title("Purchase Amount per User")

plt.tight_layout()
plt.show()


### 2-4) 연령(Age) 별 구매품목(평균) / 구매금액(평균) / 구매금액(합)


In [ ]:
user_stat_age_df = user_stat_df.groupby("Age").aggregate({
    "Product_Kinds": "mean",
    "Purchase_Amount": ["sum", "mean"]
})
user_stat_age_df.columns = ["Product_Kinds_mean", "Purchase_Amount_sum", "Purchase_Amount_mean"]
user_stat_age_df["Product_Kinds_mean"] = user_stat_age_df["Product_Kinds_mean"].round(0)
user_stat_age_df["Purchase_Amount_mean"] = user_stat_age_df["Purchase_Amount_mean"].round(0)
user_stat_age_df


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 4))

user_stat_age_df["Product_Kinds_mean"].plot(kind="bar", ax=axs[0], title="Avg Product Kinds by Age")
axs[0].set_xlabel("Age")

user_stat_age_df["Purchase_Amount_mean"].plot(kind="bar", ax=axs[1], title="Avg Purchase Amount by Age")
axs[1].yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,}'))
axs[1].set_xlabel("Age")

user_stat_age_df["Purchase_Amount_sum"].plot(kind="bar", ax=axs[2], title="Total Purchase Amount by Age")
axs[2].yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,}'))
axs[2].set_xlabel("Age")

plt.tight_layout()
plt.show()


## 3) 제품 카테고리 중심 분석

### 3-1) 요약


제품 종류는 **3631개** - 카테고리별 **2개**에서 많게는 **1047개** 임<br>
카테고리의 제품이 많을 수록 카테고리 구매액도 증가 (**상관도 0.724**)


### 3-2) 카테고리별 제품수 / 구매액(합) / 제품당 평균구매액


In [ ]:
product_cnt = df["Product_ID"].nunique()
print("제품 종류 수(Product_ID unique):", product_cnt)

product_cat_df = df.groupby("Product_Category").agg(
    Product_Kinds=("Product_ID", "nunique"),
    Total_Purchase=("Purchase", "sum")
)

product_cat_df["Avg_Purchase_per_Product"] = (
    product_cat_df["Total_Purchase"] / product_cat_df["Product_Kinds"].replace({0: np.nan})
).round(0)

product_cat_df = product_cat_df.sort_values("Total_Purchase", ascending=False)
product_cat_df


In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 5))

product_cat_df["Product_Kinds"].plot(kind="bar", ax=axs[0], title="Product Kinds by Category")
axs[0].set_xlabel("Category")
axs[0].tick_params(axis="x", rotation=0)

product_cat_df["Total_Purchase"].plot(kind="bar", ax=axs[1], title="Total Purchase by Category")
axs[1].yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,}'))
axs[1].set_xlabel("Category")
axs[1].tick_params(axis="x", rotation=0)

product_cat_df["Avg_Purchase_per_Product"].sort_values(ascending=False).plot(kind="bar", ax=axs[2], title="Avg Purchase per Product")
axs[2].yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,}'))
axs[2].set_xlabel("Category")
axs[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


### 3-3) 상관관계


In [ ]:
product_cat_df.corr(numeric_only=True)


### 3-4) 나이별 구매액 분포가 상이한 제품 카테고리

(Notebook의 stat_cat_age 분석을 정리)


In [ ]:
# 나이별 구매 비중 분포가 상이한 제품 카테고리 (10, 12, 17, 18)
fig, axs = plt.subplots(2, 2, figsize=(10, 8))

high_categories = [10, 12, 17, 18]

stat_cat_age = (df.groupby(["Age"]).size() / df.shape[0]).to_frame()
stat_cat_age.columns = ["All Categories"]

for i in range(0, 4):
    ix, iy = divmod(i, 2)
    cat = high_categories[i]

    cat_column = f"Category {cat}"
    df_catN = df[df["Product_Category"] == cat]
    stat_cat_age[cat_column] = df_catN.groupby(["Age"]).size() / df_catN.shape[0]

    cols = ["All Categories", cat_column]
    axs[ix, iy].plot(stat_cat_age.index, stat_cat_age[cols], label=cols)
    axs[ix, iy].set_title(cat_column)
    axs[ix, iy].legend()

plt.tight_layout()
plt.show()

stat_cat_age


#### stat_cat_age.describe() 중 평균(mean) / 표준편차(std)


In [ ]:
stat_cat_age[["All Categories"] + [f"Category {c}" for c in high_categories]].describe().loc[["mean", "std"]]
